# ChefBot

In [1]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel # eftersom nvidia inte finns som model i pydantic_ai
from pydantic_ai.providers.openai import OpenAIProvider
import os
from dotenv import load_dotenv

load_dotenv()

model = OpenAIChatModel(
    model_name="nvidia/nemotron-3.5-lightning-30b-a3b", # hämta från nvidia - models - den modell du vill använda
    provider=OpenAIProvider(
        base_url = "https://integrate.api.nvidia.com/v1", # finns i modelen i Nvidia
        api_key=os.environ["NVIDIA_API"]), # api nyckeln från nvidia (i .env)
)
model

OpenAIChatModel(model_name='nvidia/nemotron-3.5-lightning-30b-a3b')

In [2]:
chat_agent = Agent(
    model = model, 
    instructions=("You are a professional chef that is very good at making asian food and explaining in a pedagogical way how to create yummy dishes."
                """
                Answer in the following format:
                Description 1-2 sentences of what the dish is
                List of ingredients in bullet points
                Short step by step description on how to make the dish
                """ 
    ),
)

result = await chat_agent.run("Give me 3 asian soups in a list")

print(result.output)

Here are three iconic Asian soups, each presented in the requested format:

**Tom Yum Goong (Thai Hot-and-Sour Shrimp Soup)**  
A vibrant Thai classic celebrated for its explosive balance of spicy, salty, sour, and aromatic notes, built on a fragrant broth of lemongrass, galangal, and kaffir lime.  
- Lemongrass stalks  
- Galangal root  
- Kaffir lime leaves  
- Fish sauce  
- Fresh lime juice  
- Red chili peppers  
- Peeled and deveined shrimp  
- Sliced mushrooms  
- Fresh cilantro for garnish  

**Steps:**  
1. Bruise the lemongrass, galangal, and kaffir lime leaves and add to a pot of boiling water; simmer 5 minutes.  
2. Add shrimp and mushrooms, cooking just until the shrimp turn opaque.  
3. Stir in fish sauce, lime juice, and sliced chili; adjust seasoning to taste.  
4. Ladle into bowls, garnish with cilantro, and serve hot with jasmine rice.

---

**Miso Soup (Japanese)**  
A light, umami‑rich staple that relies on the delicate synergy between smooth dashi broth and ferment

### Make into markdown

In [3]:
with open("soups.md", "w") as file:
    file.write(result.output)

UnicodeEncodeError: 'charmap' codec can't encode character '\u2011' in position 936: character maps to <undefined>

## Task 
- change instuctions so that it outputs a python list

In [ ]:
chat_agent = Agent(
    model = model, 
    instructions=("You are a professional chef that is very good at making asian food and explaining in a pedagogical way how to create yummy dishes."
                """
                Answer in the following format:
                Description 1-2 sentences of what the dish is
                List of ingredients in bullet points
                Short step by step description on how to make the dish
                Make answer into a python list
                """ 
    ),
)

result = await chat_agent.run("Give me 3 asian soups in a list, as a python list")

print(result.output)

```python
[
    {
        "description": "A hot and sour Thai soup bursting with aromatic lemongrass, kaffir lime, and spicy chilies, typically made with shrimp or chicken.",
        "ingredients": ["lemongrass stalks", "kaffir lime leaves", "fish sauce", "lime juice", "bird's eye chilies", "oyster mushrooms", "shrimp or chicken"],
        "steps": [
            "Simmer lemongrass, lime leaves, and water for 10 minutes to extract flavor.",
            "Add protein and mushrooms, cook until nearly done.",
            "Stir in lime juice and fish sauce, adjust seasoning, and serve hot."
        ]
    },
    {
        "description": "A comforting Japanese soup featuring dashi stock, miso paste, and optional additions like tofu, wakame, and green onions.",
        "ingredients": ["dashi stock", "white miso paste", "silken tofu", "wakame seaweed", "green onions", "soy sauce (optional)"],
        "steps": [
            "Warm the dashi stock in a pot without boiling.",
            "Dissolve m

## Thinking level

In [7]:
from pydantic_ai import ModelSettings

storyteller = Agent(
    model, 
    instructions="You are a children storyteller, answer max 3 sentences",
    model_settings=ModelSettings(thinking="low", temperature=0)
    )
prompt = "A rabbit called Bella likes to program Python ..."

story1 = await storyteller.run(prompt)

print(story1.output)

Bella the rabbit loved writing Python code, especially when she could make colorful carrots pop up on her screen. One day, she built a magical garden where bunnies could hop through loops of sparkling flowers. Now, every night, her programs fill the forest with happy, hopping friends and glowing lights.


In [8]:
story1.usage

RunUsage(details={'reasoning_tokens': 373}, input_tokens=38, output_reasoning_tokens=373, output_tokens=438, requests=1)

In [9]:
story1.all_messages

<bound method AgentRunResult.all_messages of AgentRunResult(output='Bella the rabbit loved writing Python code, especially when she could make colorful carrots pop up on her screen. One day, she built a magical garden where bunnies could hop through loops of sparkling flowers. Now, every night, her programs fill the forest with happy, hopping friends and glowing lights.')>

In [10]:
story2 = await storyteller.run(prompt, timeout=60)

print(story2.output)

Bella the rabbit loved hopping through the meadow, but her favorite thing was writing cozy Python programs under the moonlight. One day, she coded a magical recipe book that could turn pebbles into delicious carrot cakes for her friends. With a cheerful `print("Yummy!")`, her garden became the most popular spot in the whole forest.


## Multimodal

In [13]:
model = OpenAIChatModel(
    model_name="nvidia/nemotron-3.5-super-vl-120b-a12b", # hämta från nvidia - models - den modell du vill använda
    provider=OpenAIProvider(
        base_url = "https://integrate.api.nvidia.com/v1", # finns i modelen i Nvidia
        api_key=os.environ["NVIDIA_API"]), # api nyckeln från nvidia (i .env)
)
model



OpenAIChatModel(model_name='nvidia/nemotron-3.5-super-vl-120b-a12b')

In [14]:
from pathlib import Path
from pydantic_ai.messages import BinaryContent

image_joker = Agent(
    model=model,
    instructions="You are a person that can analyze an image, explain the contents and make a joke about it, max 3 sentences."
)

image_data = Path("bella.png").read_bytes()
prompt = " tell me about this dude"

result = await image_joker.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type="image/png")]
)

result.output

'This grey bunny is sound asleep while wearing a tiny sailor hat and a blue and yellow ribbon labeled "Festive." He looks like he\'s ready for a ship voyage, as long as the destination is his bed. I guess you could say he\'s officially enlisted in the Navy... of Napping.'